# Taller Abandono de producto financiero

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from scipy.stats import chi2_contingency

pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

## 1. Estructura de los datos

In [2]:
df_raw = pd.read_csv("abandono_producto_financiero.csv", encoding="utf-8")
df_raw.head()

,numero_fila,id_cliente,apellido,puntaje_crediticio,pais,sexo,edad,antiguedad,saldo,numero_productos,tiene_tarjeta,miembro_activo,salario_estimado,abandono
0,1,15634602,Hargrave,619,Francia,Mujer,42,2,0.0000,1,1,1,"101,348.8800",1
1,2,15647311,Hill,608,España,Mujer,41,1,"83,807.8600",1,0,1,"112,542.5800",0
2,3,15619304,Onio,502,Francia,Mujer,42,8,"159,660.8000",3,1,0,"113,931.5700",1
3,4,15701354,Boni,699,Francia,Mujer,39,1,0.0000,2,0,0,"93,826.6300",0
4,5,15737888,Mitchell,850,España,Mujer,43,2,"125,510.8200",1,1,1,"79,084.1000",0


In [3]:
print("Columnas:", list(df_raw.columns))
print("Dimensión (filas, columnas):", df_raw.shape)
df_raw.dtypes

Columnas: ['numero_fila', 'id_cliente', 'apellido', 'puntaje_crediticio', 'pais', 'sexo', 'edad', 'antiguedad', 'saldo', 'numero_productos', 'tiene_tarjeta', 'miembro_activo', 'salario_estimado', 'abandono']
Dimensión (filas, columnas): (10000, 14)


numero_fila             int64
id_cliente              int64
apellido                  str
puntaje_crediticio      int64
pais                      str
sexo                      str
edad                    int64
antiguedad              int64
saldo                 float64
numero_productos        int64
tiene_tarjeta           int64
miembro_activo          int64
salario_estimado      float64
abandono                int64
dtype: object

In [4]:
df_raw.describe()

,numero_fila,id_cliente,puntaje_crediticio,edad,antiguedad,saldo,numero_productos,tiene_tarjeta,miembro_activo,salario_estimado,abandono
count,"10,000.0000","10,000.0000","10,000.0000","10,000.0000","10,000.0000","10,000.0000","10,000.0000","10,000.0000","10,000.0000","10,000.0000","10,000.0000"
mean,"5,000.5000","15,690,940.5694",650.5288,38.9218,5.0128,"76,485.8893",1.5302,0.7055,0.5151,"100,090.2399",0.2037
std,"2,886.8957","71,936.1861",96.6533,10.4878,2.8922,"62,397.4052",0.5817,0.4558,0.4998,"57,510.4928",0.4028
min,1.0000,"15,565,701.0000",350.0000,18.0000,0.0000,0.0000,1.0000,0.0000,0.0000,11.5800,0.0000
25%,"2,500.7500","15,628,528.2500",584.0000,32.0000,3.0000,0.0000,1.0000,0.0000,0.0000,"51,002.1100",0.0000
50%,"5,000.5000","15,690,738.0000",652.0000,37.0000,5.0000,"97,198.5400",1.0000,1.0000,1.0000,"100,193.9150",0.0000
75%,"7,500.2500","15,753,233.7500",718.0000,44.0000,7.0000,"127,644.2400",2.0000,1.0000,1.0000,"149,388.2475",0.0000
max,"10,000.0000","15,815,690.0000",850.0000,92.0000,10.0000,"250,898.0900",4.0000,1.0000,1.0000,"199,992.4800",1.0000


In [5]:
df = df_raw.drop(columns=["numero_fila", "id_cliente", "apellido"]).copy()

df["pais"] = pd.Categorical(df["pais"], categories=["Francia", "Alemania", "España"])
df["sexo"] = pd.Categorical(df["sexo"], categories=["Mujer", "Hombre"])
df["abandono"] = df["abandono"].astype(int)

print("Dimensión para el modelo:", df.shape)
print("Datos faltantes:", df.isna().sum().sum())
df.dtypes

Dimensión para el modelo: (10000, 11)
Datos faltantes: 0


puntaje_crediticio       int64
pais                  category
sexo                  category
edad                     int64
antiguedad               int64
saldo                  float64
numero_productos         int64
tiene_tarjeta            int64
miembro_activo           int64
salario_estimado       float64
abandono                 int64
dtype: object

**1. ¿Cuántas variables tiene la base cruda? ¿Cuántas quedan para el modelo?**
La base cruda tiene **14 variables** y 10.000 filas. Al eliminar los identificadores (`numero_fila`, `id_cliente`, `apellido`) quedan **11 variables**: 10 explicativas y la respuesta `abandono`.

**2. ¿Cuántas son cuantitativas?**
**6**: `puntaje_crediticio`, `edad`, `antiguedad`, `saldo`, `numero_productos` (entero) y `salario_estimado`.

**3. ¿Cuántas son categóricas o binarias (incluyendo abandono)?**
**5**: dos categóricas (`pais`, `sexo`) y tres binarias 0/1 (`tiene_tarjeta`, `miembro_activo` y la respuesta `abandono`).

**4. ¿Qué implica esta mezcla para el logit en Python?**
Las numéricas y las binarias 0/1 entran directo en la fórmula. Las categóricas de texto necesitan dummies: con `C(pais)` y `C(sexo)`, `statsmodels` (vía patsy) crea automáticamente $k-1$ dummies por variable y omite la categoría de referencia (la primera de la lista: Francia y Mujer), lo que evita la trampa de las variables ficticias (colinealidad perfecta con el intercepto). Hacer las dummies a mano obliga a crear y excluir columnas manualmente, con riesgo de error; con `C(·)` y los niveles fijados como `category`, las referencias quedan controladas y los coeficientes se leen como "diferencia frente a Francia / frente a Mujer".